<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3: Moving from pandas to Polars

This notebook accompanies Chapter 3 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

If you already know pandas, this chapter is for you. pandas and Polars look similar on the surface, but they are built on different ideas. The fastest way to become productive in Polars is to **unlearn a few pandas habits** instead of translating your pandas code line by line.

By the end of this notebook you will be able to:

- recognize what pandas and Polars have in common
- read the differences in how the two libraries *look* (code and output)
- explain five pandas concepts you can forget: the **Index**, **axes**, **bracket indexing**, **eagerness**, and **relaxedness**
- translate common operations between the two libraries
- move data back and forth between pandas and Polars

Every comparison is a **runnable example** so you can see the behavior for yourself.

**How to use this notebook:** run the cells from top to bottom. Some cells *intentionally* trigger errors to show how the libraries differ.

## 0. Setup

We install Polars, pinned to the **2.1 series**, so that everyone sees the same behavior. The `graph` extra installs what Polars needs to draw query plans (we use that in the *Eagerness* section).

In [ ]:
%pip install -q "polars[graph]==2.1.*"

In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl


print(f"pandas {pd.__version__}")
print(f"Polars {pl.__version__}")

**Convention:** pandas is imported as `pd` and Polars as `pl`. That's the one place where Polars asks you to change your habits right away: instead of starting everything with `pd.`, you start with `pl.`.

## Animals

Throughout this chapter we use a small CSV file about ten animals and some of their properties (habitat, lifespan, weight, and so on). It is small enough to read completely, but it has two deliberately awkward details:

- the **narwhal's weight is missing** (the line ends with a comma)
- the **ray's status is an empty quoted string** (`""`)

Both libraries must decide how to interpret these. Keep them in mind; we come back to them below.

The `curl` flags `-fsSL` make the download fail loudly on HTTP errors, rather than saving an error page as if it were the data.

In [ ]:
%%bash
set -euo pipefail

curl -fsSL -o sample_data/animals.csv \
    https://raw.githubusercontent.com/davidrpugh/data-analytics-with-polars/refs/heads/main/data/animals.csv

In [ ]:
ANIMALS_CSV = Path("sample_data/animals.csv")

print(ANIMALS_CSV.read_text())

## Similarities to Recognize

At a high level, the two libraries have a lot in common. Both:

- are Python packages for working with structured (tabular) data
- use the **DataFrame** as their main data structure
- build a DataFrame from **Series** (columns). Each Series has a single data type, and all Series in a DataFrame have the same length
- support the usual data types: Booleans, integers, floats, strings, categoricals, dates, times, and durations
- have a very large API
- read and write many file formats and databases
- display nicely as HTML in notebooks

That means many concepts transfer directly, and you don't start from zero.

> **What's in a name?** "pandas" comes from *panel data*, not from the animal. "Polars" was chosen because a polar bear is stronger than a panda, and because the name ends in "rs", a nod to Rust, the language Polars is written in.

## Appearances to Appreciate

Your first impression of a library comes from how its code and its output *look*. Polars will look slightly unfamiliar at first, so let's look at the differences head on.

### Differences in Code

We read the same file with both libraries. Even though the function has the same name, `read_csv`, several **argument names differ** (`sep` vs. `separator`, `header=0` vs. `has_header=True`), and the functions return different types. These small differences are among the most common sources of friction when you switch, and the documentation is your friend.

In [ ]:
animals_pd = pd.read_csv(ANIMALS_CSV, sep=",", header=0)
animals_pl = pl.read_csv(ANIMALS_CSV, separator=",", has_header=True)

print(f"{type(animals_pd) = }")
print(f"{type(animals_pl) = }")

**Both DataFrames are made of typed columns.** pandas calls them `dtypes`; Polars calls them the `schema`. Notice how each library typed the `weight` column. pandas uses a floating-point type (so that it can store `NaN`), while Polars keeps it an integer. We explain why in the *Relaxedness* section.

In [ ]:
print("pandas dtypes:")
print(animals_pd.dtypes)

In [ ]:
print("Polars schema:")
print(animals_pl.schema)

#### How did each library read the two awkward values?

The chapter asks you to guess how each library reads the missing weight and the empty status. Let's find out. For each column we count (a) missing values and (b) empty strings.

In Polars, missing data is always `null`, and an empty string is a *real value*, just not a useful one. So the ray's `""` status is **not** counted as missing. pandas turns both into `NaN`.

In [ ]:
print("pandas: number of missing values per column")
print(animals_pd.isna().sum())

In [ ]:
animals_pl.select(
    weight_nulls=pl.col("weight").null_count(),
    status_nulls=pl.col("status").null_count(),
    status_empty_strings=(pl.col("status") == "").sum(),
)

This matters later: it is the reason the "fish" group appears in Polars aggregations but disappears in pandas by default. (Spoiler: it isn't *only* about `NaN` vs. `null`; see the *Relaxedness* section.)

**Best practice:** when reading data, tell Polars up front which strings mean "missing" (the `null_values` argument of `read_csv`, covered in Chapter 6) instead of cleaning them up afterwards.

### Differences in Display

In a notebook, both libraries render DataFrames as HTML tables, but they show different information. Compare the two tables below:

- Polars shows the **shape** (rows × columns); pandas does not.
- pandas shows an **Index** (the numbers on the left); Polars has none.
- Polars shows each column's **data type** under its name.
- Missing values appear as `NaN` in pandas and `null` in Polars.
- Polars puts quotes around strings in the HTML view.

In [ ]:
animals_pd

In [ ]:
animals_pl

In a terminal (or when you call `print`), you get a *text* representation instead. Wide tables are handled differently: pandas prints as many columns as fit and continues below (so the Index appears twice), while Polars wraps text inside cells and truncates long values with `…`. Both can be configured; here we ask Polars to show all columns.

In [ ]:
print(animals_pd)

In [ ]:
with pl.Config(tbl_cols=-1):
    print(animals_pl)

A single column behaves like a one-dimensional table. In pandas, a Series shows its Index and a `dtype` (note `object` for text in pandas 2; pandas 3 shows a dedicated string type). In Polars, a Series shows its name, shape, and type.

We use `get_column` to extract a column as a Series in Polars. We'll see in a moment why we avoid the bracket syntax.

In [ ]:
animals_pd["animal"]

In [ ]:
animals_pl.get_column("animal")

To keep the following tables small we drop three columns. Notice two differences:

- pandas needs the keyword `columns=[...]`, because by default `drop` works on *rows* (more on that in the *Axes* section).
- Polars' `drop` always drops columns, and takes the names directly.

**Why new variable names?** Both libraries return a *new* DataFrame here. We give the results new names (`..._small`) instead of overwriting `animals_pd` and `animals_pl`. That way you can re-run this cell as often as you like (dropping columns that are already gone would fail), and the full DataFrames stay available.

In [ ]:
animals_pd_small = animals_pd.drop(columns=["habitat", "diet", "features"])
animals_pd_small

In [ ]:
animals_pl_small = animals_pl.drop("habitat", "diet", "features")
animals_pl_small

Some of these differences are purely cosmetic and configurable (borders, quotes). Others point to deeper conceptual differences, such as the Index. That's the next topic.

## Concepts to Unlearn

Unlearning is the hardest part of switching. These concepts shape how you *think* about working with a DataFrame, and they define the API you get. If you don't let go of them, you'll write Polars code that is slow and unidiomatic, and you'll be at a disadvantage compared to someone starting fresh.

We cover five: the Index, axes, indexing and slicing, eagerness, and relaxedness.

### Index

Every pandas DataFrame has an **Index** (sometimes a **MultiIndex**) that labels the rows. Most of the time it's a simple `RangeIndex`:

In [ ]:
animals_pd_small.index

Certain operations, such as aggregation, **move columns into the Index**. Below, the group keys `class` and `status` become a MultiIndex, so they are no longer ordinary columns:

In [ ]:
animals_agg_pd = animals_pd_small.groupby(["class", "status"])[["weight"]].mean()
animals_agg_pd

In [ ]:
animals_agg_pd.index

Many pandas users dislike this and call `reset_index()` to turn the Index back into columns:

In [ ]:
animals_agg_pd.reset_index()

**Polars has no Index at all, only columns.** The group keys simply stay columns, and the result is directly usable. We add `.sort(...)` because Polars processes groups in parallel and does **not** guarantee a particular group order. Sorting makes the output reproducible.

This also means a whole family of pandas methods has no Polars equivalent, because they only make sense with an Index: `align`, `droplevel`, `reindex`, `rename_axis`, `reset_index`, `set_axis`, `set_index`, `sort_index`, `stack`, `swaplevel`, and `unstack`.

Look closely at the output: Polars has a **row for `fish`**, and pandas above did not. Why that happens is covered in *Relaxedness* below.

In [ ]:
animals_pl_small.group_by(["class", "status"]).agg(pl.col("weight").mean()).sort("class", "status")

### Axes

pandas DataFrames have two **axes** (rows and columns), and many methods (`drop`, `dropna`, `rename`, `shift`, ...) take an `axis` argument that decides which one they work on. The default is *rows*, which is easy to forget.

For example, trying to drop the column `weight` without saying `axis=1` makes pandas look for a *row* labelled `"weight"`.

In [ ]:
animals_pd_small.drop("weight")

In [ ]:
animals_pd_small.drop("weight", axis=1)

**In Polars you never specify an axis.** Each method has one job: some always work on columns (`drop`, `rename`, `select`), others always on rows (`filter`, `sort`, `drop_nulls`).

In [ ]:
animals_pl_small.drop("weight")

### Indexing and Slicing

Square brackets are everywhere in pandas. `df[...]` can select columns, slice rows, filter rows, create or update columns, and extract values, depending on what you put inside. On top of that there are `.loc[]`, `.iloc[]`, `.at[]`, and `.iat[]`. The overlap between these tools is confusing.

**In Polars the rule is: don't use brackets.** Dedicated methods do each job (`select`, `filter`, `head`, `with_columns`). This makes code more readable and, in lazy mode, lets Polars *optimize* your query. Brackets are acceptable only for one-off inspection.

Here are the pandas patterns next to their Polars equivalents.

**Selecting columns:**

In [ ]:
animals_pd_small[["animal", "class"]]

In [ ]:
animals_pl_small.select("animal", "class")

**Filtering rows:**

In [ ]:
animals_pd_small[animals_pd_small["status"] == "endangered"]

In [ ]:
animals_pl_small.filter(pl.col("status") == "endangered")

**The first three rows:**

In [ ]:
animals_pd_small[:3]

In [ ]:
animals_pl_small.head(3)

**Updating a column.** This is where the two libraries behave very differently.

In pandas, assigning to `df["weight"]` **modifies the DataFrame in place**. To avoid corrupting `animals_pd_small` (the cells below reuse it), we first make a copy and mutate that.

In [ ]:
animals_pd_demo = animals_pd_small.copy()
animals_pd_demo["weight"] = animals_pd_demo["weight"] * 1000
animals_pd_demo[["animal", "weight"]].head(3)

In Polars, `with_columns` **returns a new DataFrame** and leaves the original alone. You describe the new column with an *expression* (`pl.col("weight") * 1000`). A reusable name for the original stays valid. Check that `animals_pl_small` is unchanged:

In [ ]:
animals_pl_grams = animals_pl_small.with_columns(pl.col("weight") * 1000)

print("New DataFrame:")
print(animals_pl_grams.select("animal", "weight").head(3))
print("\nOriginal is unchanged:")
print(animals_pl_small.select("animal", "weight").head(3))

**A subtle pandas surprise: Index alignment.** When you assign a Series to a column, pandas **aligns it on the Index**. Here we sort the `weight` column and assign it back. You'd expect the weights to be reordered, but nothing changes, because each sorted value is matched back to its original row label. We paid for a sort and got nothing.

In [ ]:
animals_pd_align = animals_pd_small.copy()
animals_pd_align["weight"] = animals_pd_align["weight"].sort_values()
animals_pd_align["weight"]

Polars has no Index, so there is no hidden alignment: it does exactly what you say. Sorting only the `weight` column reorders that column *independently* of the others, so the weights no longer line up with the animals. That is rarely what you want. To sort the *rows*, sort the DataFrame, which keeps each row's values together:

In [ ]:
# Sorts only the weight column, so the values no longer match their animal.
animals_pl_small.with_columns(pl.col("weight").sort()).head(4)

In [ ]:
# Sorts whole rows (what you usually want).
animals_pl_small.sort("weight").head(4)

### Eagerness

In pandas, **every operation runs immediately** (*eager*), and many modify objects *in place*. In Polars, operations return new objects, and you can choose between two modes:

- **Eager:** a `DataFrame`, where each step runs immediately. This is similar to the pandas experience and is great for exploration.
- **Lazy:** a `LazyFrame`, where you build a *plan* and nothing runs until you call `.collect()`. Polars can then **optimize** the whole plan first.

The query below reads the CSV lazily with `scan_csv` (which does *not* read the file yet), computes the mean weight per class, and keeps only the `mammal` row. No data has been touched by the end of this cell.

In [ ]:
lazy_query = (
    pl.scan_csv(ANIMALS_CSV)
    .group_by("class")
    .agg(pl.col("weight").mean())
    .filter(pl.col("class") == "mammal")
)
lazy_query

A **query plan** lists the steps Polars will run. Read it **from the bottom up**. We print the plan as text with `explain()`, which needs no extra tools. First the *naive* plan, exactly as written (the filter happens last):

In [ ]:
print(lazy_query.explain(optimized=False))

Now the **optimized** plan. Polars notices that filtering on `class` *before* aggregating gives the same answer (because `class` is the grouping key) while processing less data. Look for the filter appearing at the scan step, and for the scan reading only 2 of the 8 columns.

In [ ]:
print(lazy_query.explain())

The plan can also be drawn as a diagram with `show_graph()`.

In [ ]:
lazy_query.show_graph()

Finally, `collect()` executes the plan and returns a DataFrame:

In [ ]:
lazy_query.collect()

**Laziness also makes Polars fail fast.** Because Polars knows the schema before reading any data, it can detect mistakes (such as a misspelled column) *before* doing the expensive work. Here we request a column that doesn't exist; resolving the schema raises an error without reading the file.

In [ ]:
pl.scan_csv(ANIMALS_CSV).select("not_a_column").collect_schema()

On a ten-row file the lazy API can't show its speed advantage, but it shows the idea. You'll learn much more in Chapter 5. Even in eager mode, your Polars code will look the same: a chain of methods with (ideally) **no brackets**.


### Relaxedness

The last thing to unlearn is pandas' *relaxedness*. pandas tries to be forgiving and guess what you mean. Polars is **strict**. This can feel annoying at first, but strictness gives you fast feedback, correct results, and room for optimization. Below are the most important differences. The Polars cells that might raise an error are wrapped in `try`/`except` and print whatever happens, so you can see the behavior in **your** version of Polars.

#### 1. Missing values: one `null` instead of many

pandas has several ways to represent "missing" (`None`, `NaN`, `NA`, `NaT`, ...), which depend on the data type, and it treats `NaN` and `None` the same way when asking `isna()`. Polars distinguishes **missing values** (`null`, for every data type) from **invalid numbers** (`NaN`, only for floats):

In [ ]:
values = [1.0, None, float("nan")]

print("pandas isna():  ", pd.Series(values).isna().tolist())

In [ ]:
pl.DataFrame({"x": values}).select(
    "x",
    is_null=pl.col("x").is_null(),
    is_nan=pl.col("x").is_nan(),
)

#### 2. Integers stay integers when values are missing

pandas' classic integer type cannot hold missing values, so a single missing value turns the whole column into floats. Polars supports missing values in **every** data type:

In [ ]:
print("pandas:", pd.Series([1, 2, None]).dtype)
print("Polars:", pl.Series([1, 2, None]).dtype)

#### 3. Aggregations: pandas drops missing group keys, Polars keeps them

This explains the missing `fish` row from earlier. By default, pandas' `groupby` **silently drops** rows whose group key is missing. It dropped the ray because pandas read its empty status as `NaN`. Polars keeps every group.

Importantly, this is a *default setting* in pandas, not a law. With `dropna=False` pandas keeps the group too:

In [ ]:
animals_pd_small.groupby(["class", "status"], dropna=False)[["weight"]].mean()

The difference in our data also comes from how each library *read* the file: the ray's `""` is `NaN` (missing) in pandas, but an empty string (a regular value) in Polars. If you want Polars to treat it as missing, make that explicit. The `null` group then behaves like a missing key:

In [ ]:
animals_pl_nulls = animals_pl_small.with_columns(
    status=pl.when(pl.col("status") == "").then(None).otherwise(pl.col("status"))
)

animals_pl_nulls.group_by("class", "status").agg(pl.col("weight").mean()).sort("class", "status")

#### 4. No silent fallback to Python objects

If you mix incompatible types in a pandas column, it quietly falls back to the generic `object` type, which is slow and gives up type safety. Polars never falls back to Python objects; it raises an error instead.

In [ ]:
print("pandas dtype:", pd.Series([1, "a", 2.5]).dtype)

In [ ]:
pl.Series([1, "a", 2.5])

#### 5. Column names

pandas allows column names of any type and allows **duplicate** names. In Polars, column names must be **strings** and must be **unique**. Unique names are what let Polars apply expressions to columns safely and in parallel.

In [ ]:
pd_duplicates = pd.DataFrame([[1, 2]], columns=["a", "a"])
print("pandas columns:", list(pd_duplicates.columns))

In [ ]:
pl.DataFrame({"a": [1], "b": [2]}).rename({"b": "a"})

#### 6. Dates and Booleans

pandas lets you compare dates with strings and Booleans with integers. In Polars you are expected to use proper date values (`pl.date(...)`, Python `datetime`s) and Booleans. Polars is also intentionally explicit about type conversions. Run the cell to see how your version behaves:

In [ ]:
dates_pd = pd.Series(pd.to_datetime(["2024-01-15", "2024-06-15"]))
print("pandas, comparing to a date string:", (dates_pd > "2024-03-01").tolist())


In [ ]:
dates_pl = pl.DataFrame({"d": ["2024-01-15", "2024-06-15"]}).with_columns(pl.col("d").str.to_date())
dates_pl.select("d", after_march=pl.col("d") > pl.date(2024, 3, 1))

In [ ]:
print("pandas, Boolean == 1:", (pd.Series([True, False]) == 1).tolist())

In [ ]:
print("Polars, Boolean == 1:", (pl.Series([True, False]) == 1).to_list())

#### 7. Return types are predictable

In pandas, the type of the result depends on what you pass: `.loc[0]` returns a **Series**, `.loc[0, "animal"]` returns a **scalar**, and `sum(axis=0)` returns a **Series**. In Polars, selecting and aggregating always returns a **DataFrame**. If you want a Python value, you ask for it explicitly (for example with `.item()`).

In [ ]:
print("pandas .loc[0]            ->", type(animals_pd_small.loc[0]).__name__)
print("pandas .loc[0, 'animal']  ->", type(animals_pd_small.loc[0, "animal"]).__name__)
print("pandas sum()              ->", type(animals_pd_small[["lifespan", "weight"]].sum()).__name__)

In [ ]:
# A one-row DataFrame, not a Series or a scalar.
animals_pl_small.filter(pl.col("animal") == "duck")

In [ ]:
# A one-row DataFrame again, rather than a Series.
animals_pl_small.select("lifespan", "weight").sum()

In [ ]:
# Ask explicitly for a Python value.
animals_pl_small.filter(pl.col("animal") == "duck").select("weight").item()

**The payoff for the strictness:** Polars can tell you early whether a query will fail, guarantees consistent behavior across data types, and can optimize your queries aggressively. In the authors' words, Polars being stricter is "a feature rather than a bug."

## Syntax to Forget

Once the concepts click, the rest is mostly different *names* for functions and arguments. That is annoying, but the documentation resolves it quickly.

> **Lost in translation.** It's tempting to translate your pandas code to Polars line by line. That's like translating a Dutch idiom word for word: it technically works but "puts no sods on the dike". You end up with code that isn't idiomatic and may not be fast. Learn the Polars way of expressing the *idea* instead.

### Common Operations Side By Side

Each pair of cells below does the same job. Pay attention to the *method names*, the *argument names*, and any *differences in the output*.

#### Removing duplicate values

Keep one row per `class`. pandas calls this `drop_duplicates`; Polars calls it `unique`.

**Be careful here:** pandas keeps the **first** row of each group, always, in the original order. By default Polars keeps **any** row and doesn't promise to preserve order, because it runs in parallel. To reproduce the pandas behavior exactly, ask for it. Maintaining order costs some performance, so only do it when you need it.

In [ ]:
animals_pd_small.drop_duplicates(subset="class")

In [ ]:
animals_pl_small.unique(subset="class", keep="first", maintain_order=True)

#### Removing missing values

Drop rows where `weight` is missing. `dropna` in pandas becomes `drop_nulls` in Polars. Note that the ray survives in both, because its *weight* is present (it's the *status* that is empty).

In [ ]:
animals_pd_small.dropna(subset="weight")

In [ ]:
animals_pl_small.drop_nulls(subset="weight")

#### Sorting rows

Sort by weight, heaviest first. Differences to note:

- `sort_values` (pandas) vs. `sort` (Polars)
- `ascending=False` (pandas) vs. `descending=True` (Polars). Mixing these up is a classic mistake!
- missing values come **last** in pandas but **first** in Polars by default. Use `nulls_last=True` to match pandas.

In [ ]:
animals_pd_small.sort_values("weight", ascending=False)

In [ ]:
animals_pl_small.sort("weight", descending=True)

In [ ]:
animals_pl_small.sort("weight", descending=True, nulls_last=True)

#### Casting an existing column

Convert `lifespan` from integer to float. pandas uses `astype` (typically assigned back to the column); Polars uses the `cast` method inside an **expression**, passed to `with_columns`. Expressions are the heart of Polars, and Part III of the book covers them in depth.

In [ ]:
animals_pd_small.assign(lifespan=animals_pd_small["lifespan"].astype(float))

In [ ]:
animals_pl_small.with_columns(pl.col("lifespan").cast(pl.Float64))

#### Aggregating rows

Mean weight per class and status. Differences:

- `groupby` (pandas) vs. `group_by` (Polars, with an underscore)
- pandas moves the keys into a MultiIndex; Polars keeps them as columns
- Polars' `agg` takes **expressions**, so you can build any aggregation you can express, whereas pandas is largely limited to a list of built-in functions

In [ ]:
animals_pd_small.groupby(["class", "status"])[["weight"]].mean()

In [ ]:
animals_pl_small.group_by("class", "status").agg(pl.col("weight").mean()).sort("class", "status")

### Thinking in Polars: one chain, not many steps

The same question, asked of both libraries: *"Among mammals with a known weight, what's the average weight per year of life, for each conservation status, highest first?"*

The pandas version below is written in an idiomatic **method-chaining** style, which already helps. The Polars version reads almost like the question itself: each method does one thing, expressions describe the computations, and there are no brackets or temporary variables.

In [ ]:
(
    animals_pd_small.loc[lambda df: df["class"] == "mammal"]
    .dropna(subset=["weight"])
    .assign(weight_per_year=lambda df: df["weight"] / df["lifespan"])
    .groupby("status", as_index=False)["weight_per_year"]
    .mean()
    .sort_values("weight_per_year", ascending=False)
)

In [ ]:
(
    animals_pl_small.filter(pl.col("class") == "mammal")
    .drop_nulls("weight")
    .with_columns(weight_per_year=pl.col("weight") / pl.col("lifespan"))
    .group_by("status")
    .agg(pl.col("weight_per_year").mean())
    .sort("weight_per_year", descending=True)
)

Because this is a chain of Polars operations, it could be turned into a lazy query by starting with `pl.scan_csv(...)` and ending with `.collect()`. Polars would then optimize the whole thing.

## To and From pandas

You don't have to convert everything at once. A practical way to adopt Polars is to replace *one slow step* of an existing pandas pipeline and move data between the two libraries at the boundary.

- `pl.from_pandas(df)` converts a pandas DataFrame to Polars. This is the documented, explicit conversion, and it has options such as `nan_to_null`. (Check the docs for the current defaults.)
- `df.to_pandas()` converts back.

Both conversions need `pyarrow` and `numpy`, which Colab provides.

In [ ]:
animals_pl_converted = pl.from_pandas(animals_pd_small)

In [ ]:
print(animals_pl_converted.schema)

In [ ]:
animals_pl_converted

Notice that the `weight` column is now `Float64`. It was a float in pandas (because of the `NaN`), and the conversion doesn't go back and make it an integer. The pandas `NaN` in the narwhal's row became a Polars `null`, which we confirm below.

In [ ]:
animals_pl_converted.select(weight_nulls=pl.col("weight").null_count())

Converting back to pandas gives you a regular pandas DataFrame. Missing text values come back as `None`, and missing floats as `NaN`.

In [ ]:
there_and_back_again_df = animals_pl_converted.to_pandas()
there_and_back_again_df

pandas can also use **Arrow-backed** data types, which keep real missing values (instead of `NaN`) and are closer to how Polars stores data. If you're sending data back to pandas for a library that supports it, this can avoid some type changes:

In [ ]:
animals_pl_converted.to_pandas(use_pyarrow_extension_array=True).dtypes

## Cheat sheet: pandas to Polars

| Task | pandas | Polars |
|---|---|---|
| Read CSV | `pd.read_csv(path, sep=",")` | `pl.read_csv(path, separator=",")` |
| Lazy read | n/a | `pl.scan_csv(path)` ... `.collect()` |
| Select columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filter rows | `df[df["a"] > 1]` | `df.filter(pl.col("a") > 1)` |
| First *n* rows | `df[:n]` or `df.head(n)` | `df.head(n)` |
| Add or update a column | `df["c"] = ...` or `df.assign(c=...)` | `df.with_columns(c=...)` |
| Change a type | `s.astype(float)` | `pl.col("a").cast(pl.Float64)` |
| Drop columns | `df.drop(columns=[...])` | `df.drop(...)` |
| Drop missing rows | `df.dropna(subset=...)` | `df.drop_nulls(subset=...)` |
| Drop duplicates | `df.drop_duplicates(subset=...)` | `df.unique(subset=..., keep="first", maintain_order=True)` |
| Sort rows | `df.sort_values("a", ascending=False)` | `df.sort("a", descending=True)` |
| Group and aggregate | `df.groupby("k")["v"].mean()` | `df.group_by("k").agg(pl.col("v").mean())` |
| Rename | `df.rename(columns={...})` | `df.rename({...})` |
| Convert to the other library | `pl.from_pandas(df)` | `df.to_pandas()` |

## Takeaways

- Without pandas there would be no Polars, and the two share many ideas: DataFrames, Series, typed columns, rich APIs.
- Polars output shows **shape** and **data types**, and has **no Index**.
- Forget the **Index**, **axes**, and **brackets**: Polars has one method per job, and columns stay columns.
- Polars operations return **new** DataFrames. Nothing is modified in place, so you can safely keep intermediate results.
- Polars offers an **eager** and a **lazy** mode. Lazy queries are optimized before running, and mistakes are caught before data is read.
- Polars is **strict**: one `null` for missing data, predictable types, unique string column names, no silent fallback to Python objects. That strictness buys you reliability and speed.
- Some groups, orderings, and defaults differ (`keep`, `maintain_order`, `nulls_last`, `dropna`). Check them when you want pandas-identical results.
- Moving between the libraries is easy: `pl.from_pandas()` and `.to_pandas()`.

### Next up

In Chapter 4 we look at Polars' data structures and data types in depth: Series, DataFrames, LazyFrames, and how missing data is represented.

### Try it yourself

1. Add a column `lifespan_months` to `animals_pl_small` (multiply by 12) and compute the same thing in pandas **without** mutating `animals_pd_small`.

2. Find the animal with the highest `weight` in each `class`. (Hint: `group_by` then `agg`, or a window function with `pl.col(...).over(...)`.)


3. Re-run the "Thinking in Polars" example as a **lazy** query with `pl.scan_csv(...)`. Print its `explain()` plan. What happens to the filter?


4. Convert the empty `status` string in Polars to a `null` *while reading the file* by using the `null_values` argument of `pl.read_csv`. Does the `fish` group behave differently in a `group_by`?